# Cost-Aware Workflow Example

This notebook demonstrates how to use `CostAwareSimulator` and `CostInterpModel` in BayesFlow to optimize simulation budgets by predicting and filtering for low-cost parameter samples.

## 1. Imports

In [11]:
import os
import pickle
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from bayesflow.simulators.benchmark_simulators.sir import SIR
from bayesflow.simulators.cost_aware_simulator import CostAwareSimulator
from cost_interp_model import CostInterpModel
from bayesflow.approximators import RatioApproximator, ContinuousApproximator
from bayesflow.adapters import Adapter


## 2. Generate Synthetic Cost Data

In a real scenario, cost can be any value that is meaningful for the simulator, for example wall-clock time of the simulation. In this toy example, we use random noise as a function of the SIR parameters ($\beta ,\gamma$) with some added noise.

In [13]:
def make_synthetic_cost_data(cost_data_dir="cost_data_sir"):
    # Initialize the base SIR simulator
    sir_sim = SIR()

    os.makedirs(cost_data_dir, exist_ok=True)

    print("Generating synthetic cost data...")
    num_cost_samples = 20
    for i in range(num_cost_samples):
        theta = sir_sim.prior()
        # Simulate cost: higher beta -> higher cost
        actual_cost = 0.1 + np.abs(theta[0]) + np.abs(np.random.normal(0, 0.1))
        print(actual_cost)
        with open(f"{cost_data_dir}/sample_{i}.pkl", "wb") as f:
            pickle.dump({"theta": theta, "cost": actual_cost}, f)
            
    return sir_sim


cost_data_dir = "cost_data_sir"
sir_sim = make_synthetic_cost_data(cost_data_dir)


Generating synthetic cost data...
1.183285719376362
0.7367363023644398
0.4659110283554397
0.28866989990699937
0.29330340534392046
1.084111144453854
0.5302130726364993
0.49549807201134805
0.588661557120353
0.5865407358014597
0.7797214925664124
0.9287316038229259
0.3782294703421969
0.8239556854461221
1.1327586844322568
0.35264513923719315
0.36537784219216873
0.715419299750443
0.5799105308297062
0.4568850112313214


## 3. Fit Cost Interpolation Model

We use a small test Gaussian Process (GP) called `CostInterpModel` to predict the cost given the parameters $\theta$.
This is a user input so can be very varied. 

In [16]:
print("Fitting cost interpolation model...")
cost_model = CostInterpModel(root=cost_data_dir).fit(length_scale=3.0)


Fitting cost interpolation model...


In [19]:
# take a look at the cost values
b_min, b_max = 0, 2  # Approximate ranges for SIR beta
g_min, g_max = 0, 2  # Approximate ranges for SIR gamma
B, G = np.meshgrid(np.linspace(b_min, b_max, 50), np.linspace(g_min, g_max, 50))
grid_theta = np.stack([B.ravel(), G.ravel()], axis=1)

predicted_costs, _ = cost_model.predict(grid_theta)
COST = predicted_costs.reshape(B.shape)
COST

array([[0.31581124, 0.34822977, 0.38247303, ..., 2.21517207, 2.22578162,
        2.23375201],
       [0.28353041, 0.31406367, 0.34642529, ..., 2.11397775, 2.12384872,
        2.13111883],
       [0.25570226, 0.28435977, 0.31484815, ..., 2.01672467, 2.02583274,
        2.03237807],
       ...,
       [3.10290989, 3.07039597, 3.03881155, ..., 1.83897244, 1.79870174,
        1.75714637],
       [3.22839044, 3.19521802, 3.16293923, ..., 1.91218919, 1.87077083,
        1.82808388],
       [3.3547643 , 3.32096646, 3.28802612, ..., 1.98739426, 1.94483935,
        1.9010314 ]], shape=(50, 50))

## 4. Initialize Cost-Aware Simulator

The `CostAwareSimulator` wraps the base simulator and uses the fitted cost model to filter samples based on a regularised cost function $g(c(\theta))$.

In [20]:
cost_aware_sim = CostAwareSimulator(simulator=sir_sim, cost_model=cost_model)


## 5. Cost-Aware Sampling

We use rejection sampling to obtain a set of cost-efficient parameters.

In [39]:
print("\nTesting cost-aware sampling...")
num_samples = 50

#should be just 'sample'
accepted_samples = cost_aware_sim.sample(batch_shape=(num_samples,1))

accepted_theta = accepted_samples.get("theta")
if accepted_theta is None:
    accepted_theta = accepted_samples.get("parameters")

print(f"Successfully sampled {len(accepted_theta)} cost-efficient parameters.")



Testing cost-aware sampling...
Cost Aware simulator does rejection sampling based on the cost function
Successfully sampled 50 cost-efficient parameters.


## 6. Visualization: Cost Landscape and Samples

To understand how cost-aware sampling works, we visualize the predicted cost surface and overlay the sampled parameters.

In [ ]:
# Create a grid for cost visualization
b_min, b_max = 0, 2  # Approximate ranges for SIR beta
g_min, g_max = 0, 2  # Approximate ranges for SIR gamma
B, G = np.meshgrid(np.linspace(b_min, b_max, 50), np.linspace(g_min, g_max, 50))
grid_theta = np.stack([B.ravel(), G.ravel()], axis=1)

predicted_costs, _ = cost_model.predict(grid_theta)
COST = predicted_costs.reshape(B.shape)

plt.figure(figsize=(12, 8))
cp = plt.contourf(B, G, COST, cmap="viridis", alpha=0.7)
plt.colorbar(cp, label="Predicted Cost")

# Prior samples for comparison
prior_samples = np.array([sir_sim.prior() for _ in range(500)])
plt.scatter(
    prior_samples[:, 0],
    prior_samples[:, 1],
    color="white",
    alpha=0.3,
    s=10,
    label="Prior Samples",
    marker=".",
)

# Accepted samples
if len(accepted_theta) > 0:
    plt.scatter(
        accepted_theta[:, 0],
        accepted_theta[:, 1],
        color="red",
        edgecolors="white",
        s=50,
        label="Cost-Aware Samples",
        marker="o",
    )

plt.xlabel(r"$\beta$ (Contact Rate)")
plt.ylabel(r"$\gamma$ (Recovery Rate)")
plt.title("Cost Landscape and Sample Distribution")
plt.legend()
plt.show()

# Visualise the change in parameter distributions
plt.figure(figsize=(12, 5))

# Distribution of beta
plt.subplot(1, 2, 1)
plt.hist(prior_samples[:, 0], bins=20, alpha=0.5, label="Prior", color="gray")
if len(accepted_theta) > 0:
    plt.hist(accepted_theta[:, 0], bins=20, alpha=0.7, label="Cost-Aware", color="red")
plt.xlabel(r"$\beta$")
plt.ylabel("Density")
plt.title("Distribution of $\beta$")
plt.legend()

# Distribution of gamma
plt.subplot(1, 2, 2)
plt.hist(prior_samples[:, 1], bins=20, alpha=0.5, label="Prior", color="gray")
if len(accepted_theta) > 0:
    plt.hist(accepted_theta[:, 1], bins=20, alpha=0.7, label="Cost-Aware", color="red")
plt.xlabel(r"$\gamma$")
plt.ylabel("Density")
plt.title("Distribution of $\gamma$")
plt.legend()

plt.tight_layout()
plt.show()


## 7. Performance Metrics

We evaluate the effectiveness of the cost-aware sampling using Effective Sample Size (ESS) and Computational Gain (CG) on a test batch.

In [41]:

candidates_dict = {"theta": accepted_theta}
accepted_mask = cost_aware_sim.predicate(candidates_dict)
num_accepted = np.sum(accepted_mask)

metrics = cost_aware_sim.compute_metrics(candidates_dict, accepted_mask)
print("\nPerformance Metrics (based on a test batch of 100):")
print(f"  ESS: {metrics['ess']:.2f}")
print(f"  CG:  {metrics['cg']:.2f}")



Performance Metrics (based on a test batch of 100):
  ESS: 1.00
  CG:  1.00


## 8. Running the Expensive Simulator

Only the parameters that passed the cost predicate are passed to the actual simulation.

In [11]:
results = [sir_sim.observation_model(t) for t in accepted_theta]
print(f"\nSuccessfully simulated {len(results)} samples using the expensive simulator.")



Successfully simulated 33 samples using the expensive simulator.


## 9. Training an Approximator on Cost-Efficient Samples

Finally, we show how to use these samples to train a model, using importance weights to correct the sampling bias introduced by cost-filtering.

In [ ]:
print("\nTraining an example model on cost-efficient samples...")
if num_accepted > 0:
    train_num_candidates = 500
    train_candidates_theta = np.array(
        [sir_sim.prior() for _ in range(train_num_candidates)]
    )
    train_candidates_dict = {"theta": train_candidates_theta}

    train_accepted_mask = cost_aware_sim.predicate(train_candidates_dict)
    train_accepted_theta = train_candidates_theta[train_accepted_mask]

    train_observations = np.array(
        [sir_sim.observation_model(t) for t in train_accepted_theta]
    )

    train_predicted_cost, _ = cost_model.predict(train_candidates_theta)
    train_g_val = cost_aware_sim.regularise_cost(train_predicted_cost)
    train_weights = cost_aware_sim.compute_weights(train_g_val, train_accepted_mask)

    adapter = Adapter().create_default(inference_variables=["theta"])

    train_data = {
        "theta": train_accepted_theta,
        "observations": train_observations,
        "weights": train_weights,
    }

    transformed_data = adapter(train_data)

    inference_net = nn.Sequential(
        nn.Linear(train_observations.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
    )

    approximator = ContinuousApproximator(
        inference_network=inference_net, adapter=adapter
    )

    data_shapes = {
        "inference_variables": train_accepted_theta.shape[1:],
        "inference_conditions": train_observations.shape[1:],
    }
    approximator.build(data_shapes)

    print(
        f"Model built successfully. Training on {len(train_accepted_theta)} samples with importance weights."
    )
    print("The workflow is now complete: Cost-aware sampling -> Model training.")
else:
    print("Skipping model training as no samples were accepted.")
